### Packages for the Data Generation and Mopdelling of PD (Probability of Default), LGD (Loss Given Default) and EAD (Exposure at Default)

In [1]:
# Data Management and Processing
import pandas as pd
import numpy as np
import scipy
import random

In [2]:
# Machine Learning and Statistics
import sklearn
import statsmodels.api as sm
import tensorflow as tf

2025-03-21 13:17:25.279186: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Data generator

##### Support functions

In [3]:
############# Function to create a profession based on the educational level #########################
def generate_profession(education):
    if education == "high school or lower":
        return random.choices(["LowSkilled", "Unemployed_LowSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education == "ausbildung":
        return random.choices(["MediumSkilled", "Unemployed_MediumSkilled"], weights=[0.9, 0.1], k=1)[0]
    if education in ["bachelor degree", "post graduate degree"]:
        return random.choices(["HighSkilled", "Unemployed_HighSkilled"], weights=[0.9, 0.1], k=1)[0]

In [15]:
############################### Function to generate monthly income and expenditure ##################################                

# Define income parameters for different profession levels and age ranges
income_parameters = {
    ("LowSkilled", "Unemployed_LowSkilled"): {
        (30, 35): {"mean": 1000, "std_dev": 200, "max_income": 2000},
        (36, 40): {"mean": 1200, "std_dev": 200, "max_income": 2400},
        (41, 45): {"mean": 1500, "std_dev": 250, "max_income": 3000},
        (46, 50): {"mean": 1800, "std_dev": 300, "max_income": 3600},
        (51, 55): {"mean": 2000, "std_dev": 350, "max_income": 4000},
        (56, 60): {"mean": 2200, "std_dev": 400, "max_income": 4300},
        (61, 65): {"mean": 2400, "std_dev": 600, "max_income": 4500},
    },
    ("MediumSkilled", "Unemployed_MediumSkilled"): {
        (30, 35): {"mean": 1800, "std_dev": 300, "max_income": 4000},
        (36, 40): {"mean": 2300, "std_dev": 400, "max_income": 5000},
        (41, 45): {"mean": 2600, "std_dev": 500, "max_income": 6000},
        (46, 50): {"mean": 3000, "std_dev": 600, "max_income": 7000},
        (51, 55): {"mean": 3500, "std_dev": 800, "max_income": 8000},
        (56, 60): {"mean": 4000, "std_dev": 800, "max_income": 9000},
        (61, 65): {"mean": 5000, "std_dev": 1000, "max_income": 10000},
    },
    ("HighSkilled", "Unemployed_HighSkilled"): {
        (30, 35): {"mean": 3000, "std_dev": 500, "max_income": 10000},
        (36, 40): {"mean": 4500, "std_dev": 700, "max_income": 15000},
        (41, 45): {"mean": 6000, "std_dev": 1000, "max_income": 20000},
        (46, 50): {"mean": 7000, "std_dev": 1500, "max_income": 25000},
        (51, 55): {"mean": 8000, "std_dev": 2000, "max_income": 30000},
        (56, 60): {"mean": 9000, "std_dev": 3000, "max_income": 40000},
        (61, 65): {"mean": 10000, "std_dev": 4000, "max_income": 50000},
    }
}

# To calculate the monthly income and expenditure
def generate_income_expense(profession_undertake, current_age, num_dependents):
    # Iterate over income parameters for each profession group
    for profession_group, age_ranges in income_parameters.items():
        # Check if profession_undertake is one of the professions in the profession_group tuple
        if profession_undertake in profession_group:
            # Iterate over the age ranges and income parameters
            for age_range, params in age_ranges.items():
                if age_range[0] <= current_age <= age_range[1]:
                    mean_income = params["mean"]
                    std_dev = params["std_dev"]
                    max_income = params["max_income"]
                    unemployement_money = mean_income * 0.5  # 50% of mean income for unemployment

                    # If profession_undertake contains the word "Unemployed" before "_", return the unemployment money
                    if profession_undertake.split("_")[0] == "Unemployed":
                        income = unemployement_money
                        expenditure = generate_expenditure(income, mean_income, num_dependents) # The belong to the population under mean_income
                        return income, expenditure
                    
                    # If profession_undertake does not contain the word "Unemployed", generate income with an specific rule
                    else:
                        calc_income = int(np.random.normal(mean_income, std_dev))
                        income = max(unemployement_money, min(calc_income, max_income))  
                        expenditure = generate_expenditure(income, mean_income, num_dependents)
                        return income, expenditure

# Support function to calculate the expenditure
def generate_expenditure(income, mean_income, num_dependents):
    # Values for low and for high income people (lower possible value, mode, higher possible value) depending on the number of dependents
    low_income_params = [(0.5, 0.7, 1.5), (0.7, 0.8, 1.5), (0.8, 0.9, 1.5), (0.9, 0.9, 1.5), (0.9, 1.0, 1.5)]
    high_income_params = [(0.5, 0.7, 1.5), (0.6, 0.7, 1.5), (0.7, 0.75, 1.5), (0.7, 0.75, 1.5), (0.8, 0.85, 1.5)]
    # The parameters that should be taken depend on whether the income is below or above the mean income
    params = low_income_params if income < mean_income else high_income_params
    # Here we recover the parameters
    left, mode, right = params[min(num_dependents, 4)]  # Ensure index stays within range
    # The expenditure is calculated given a rule of min, mode, max
    expenditure = income * np.random.triangular(left=left, mode=mode, right=right)
    return expenditure



In [16]:
############################### Function to generate the credit to be requested ##################################

In [17]:
############################### Function to generate the Y-variable (default 0, 1) ##################################     
def default_y_calculation(profession, debt_income_ratio):
    # past_credits, dependents, professon, debt_income_ratio
    if profession == ("Unemployed_LowSkilled" or "Unemployed_MediumSkilled") and debt_income_ratio > 0.1:
        return 1
    if profession == ("Unemployed_HighSkilled") and debt_income_ratio > 0.2:
        return 1     


##### Data Generator for the original state of individuals

In [18]:
# Function to generate data accordingly to some requirements
def data_generator(number_of_customers):
    data = []
    for i in range(number_of_customers):
        
        # ---------------- X-Variables -------------------------------#
        ##### Variables not directly dependent on other variables #####
        name = f"name{i}" # names are created according to the index "i"
        age = random.randint(30, 60) # As the maximum attainable age that we want in the game is 65
        education_level = random.choices(["high school or lower", "ausbildung", "bachelor degree", "post graduate degree"],  weights=[0.3, 0.3, 0.3, 0.1], k=1)[0]
        # Number of unpaid past credits
        past_credits = random.choices([0, 1, 2, 3], weights=[0.6, 0.3, 0.08, 0.02], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        # Number of dependents
        dependents = random.choices([0, 1, 2, 3, 4], weights=[0.6, 0.3, 0.06, 0.03, 0.01], k=1)[0] # This emphasizes 0 and 1 unpaid credits
        
        ##### Variables directly dependent on other variables #####
        # Generate profession based on education level
        profession = generate_profession(education_level)
        # Generate monthly income based on profession, age and number of dependents
        monthly_income = generate_income_expense(profession, age, dependents)[0]
        # Generate monthly expenditure dependening on the income, mean income and number of dependents
        monthly_expenditure = generate_income_expense(profession, age, dependents)[1]
        
        ##### Other variables generated from the variables above #####
        savings_debt = monthly_income - monthly_expenditure
        # Debt to income ratio
        if savings_debt < 0:
            #debt_to_income_ratio = f"{abs(savings_debt/monthly_income):.2%}"
            debt_to_income_ratio = abs(savings_debt/monthly_income)
        else:
            #debt_to_income_ratio = f"{0:.2%}"
            debt_to_income_ratio = 0
        
        # ---------------- Credit amount requested and time of the request--------------------------------#
        
        
        # ---------------- Y-Variable --------------------------------#
        
        data.append({
            'name': name,
            'age': age,
            'educational level': education_level,
            'number of not paid past credits': past_credits,
            'dependents': dependents,
            'profession': profession,
            'monthly income': monthly_income,
            'monthly expenditure': monthly_expenditure,
            'savings (debt)': savings_debt,
            'debt-to-income ratio': debt_to_income_ratio
        })

    # Create a pandas DataFrame
    df = pd.DataFrame(data)
    return df

##### Generating one data frame

In [30]:
number_of_customers_1 = 1000
df_1 = data_generator(number_of_customers_1)
df_1

,name,age,educational level,number of not paid past credits,dependents,profession,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
0,name0,37,high school or lower,1,0,LowSkilled,860.0,1440.053050,-580.053050,0.674480
1,name1,38,bachelor degree,0,0,HighSkilled,5288.0,6486.777996,-1198.777996,0.226698
2,name2,58,ausbildung,0,0,MediumSkilled,3183.0,5826.914709,-2643.914709,0.830636
3,name3,58,post graduate degree,0,0,HighSkilled,7740.0,11816.957963,-4076.957963,0.526739
4,name4,31,ausbildung,1,1,MediumSkilled,1666.0,1362.124001,303.875999,0.000000
...,...,...,...,...,...,...,...,...,...,...
995,name995,36,ausbildung,0,0,Unemployed_MediumSkilled,1150.0,1039.920965,110.079035,0.000000
996,name996,34,bachelor degree,0,0,HighSkilled,2985.0,2435.535617,549.464383,0.000000
997,name997,48,ausbildung,2,0,MediumSkilled,3146.0,2208.115394,937.884606,0.000000
998,name998,41,ausbildung,0,1,Unemployed_MediumSkilled,1300.0,977.625471,322.374529,0.000000


##### DF statistics

In [31]:
df_1.describe()

,age,number of not paid past credits,dependents,monthly income,monthly expenditure,savings (debt),debt-to-income ratio
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,45.574000,0.517000,0.547000,3675.344000,3487.787289,187.556711,0.131558
std,8.912573,0.722657,0.835757,2653.324346,2677.807265,1632.239595,0.244516
min,30.000000,0.000000,0.000000,500.000000,478.542816,-9223.388466,0.000000
25%,38.000000,0.000000,0.000000,1754.000000,1592.208281,-404.258969,0.000000
50%,46.000000,0.000000,0.000000,2789.000000,2616.713043,156.612486,0.000000
75%,53.000000,1.000000,1.000000,4729.000000,4528.229935,748.143642,0.171407
max,60.000000,3.000000,4.000000,13698.000000,18569.388466,8166.226394,1.971354


Monthly income by profession

In [24]:
df_1.groupby('profession')['monthly income'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,359.0,6117.576602,2529.114621,1989.0,4240.50,5555.0,7730.5,14112.0
LowSkilled,274.0,1599.215328,534.617909,500.0,1151.75,1556.0,1992.0,3146.0
MediumSkilled,265.0,2882.618868,948.343381,1286.0,2111.00,2724.0,3548.0,5645.0
Unemployed_HighSkilled,40.0,3056.250000,1067.418855,1500.0,2250.00,3000.0,4000.0,4500.0
Unemployed_LowSkilled,27.0,831.481481,203.879748,500.0,600.00,900.0,1000.0,1100.0
Unemployed_MediumSkilled,35.0,1290.000000,353.927543,900.0,900.00,1150.0,1500.0,2000.0


Debt-to-income ratio by profession

In [25]:
df_1.groupby('profession')['debt-to-income ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
profession,,,,,,,,
HighSkilled,359.0,0.143421,0.276640,0.0,0.0,0.0,0.174916,1.990873
LowSkilled,274.0,0.105775,0.192960,0.0,0.0,0.0,0.145133,1.109482
MediumSkilled,265.0,0.110484,0.250759,0.0,0.0,0.0,0.114568,2.395336
Unemployed_HighSkilled,40.0,0.072408,0.101579,0.0,0.0,0.0,0.150723,0.305203
Unemployed_LowSkilled,27.0,0.046222,0.087399,0.0,0.0,0.0,0.063902,0.407052
Unemployed_MediumSkilled,35.0,0.052046,0.102351,0.0,0.0,0.0,0.029806,0.383485
